# AAL Sales Analysis - Q4 2020
## Objective
Analyze Australian Apparel Ltd (AAL) sales data for Q4 2020 to identify high-revenue states and develop intervention strategies for underperforming regions.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy

# Styling and display options
sns.set_style("darkgrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1. Data Wrangling
### 1.1 Load and Validate

In [7]:
import os

# Path to CSV (relative to notebook location)
filepath = os.path.join('../data', 'AusApparalSales4thQrt2020.csv')
if not os.path.exists(filepath):
    raise FileNotFoundError(f'File not found: {filepath}')

df = pd.read_csv(filepath)

print('Shape:', df.shape)
print('Info:')
df.info()

df.head()

if df.empty:
    raise ValueError('Loaded DataFrame is empty — check the source CSV or path.')

Shape: (7560, 6)
Info:
<class 'pandas.DataFrame'>
RangeIndex: 7560 entries, 0 to 7559
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Date    7560 non-null   str  
 1   Time    7560 non-null   str  
 2   State   7560 non-null   str  
 3   Group   7560 non-null   str  
 4   Unit    7560 non-null   int64
 5   Sales   7560 non-null   int64
dtypes: int64(2), str(4)
memory usage: 354.5 KB


In [11]:
# Inspect missing / present values
na_counts = df.isna().sum()
print('Missing values per column:')
print(na_counts)

# Show a quick boolean mask (first rows) to inspect not-null entries
print()
print('Not-null entries (first 5 rows):')
df.notna().head()

Missing values per column:
Date     0
Time     0
State    0
Group    0
Unit     0
Sales    0
dtype: int64

Not-null entries (first 5 rows):


,Date,Time,State,Group,Unit,Sales
0,True,True,True,True,True,True
1,True,True,True,True,True,True
2,True,True,True,True,True,True
3,True,True,True,True,True,True
4,True,True,True,True,True,True


### 1.2 Data Cleaning and Aggregation

### 1.2.1 Data Cleaning Strategy

**Finding:** No null values detected across all 7,560 rows and 6 columns.

**Decision:** No data cleaning required.

**Justification:** 
- The dataset represents aggregated transaction summaries at (Date, Time, State, Group) grain
- Each row is a complete observation with valid Units and Sales values
- No missing combinations or invalid entries detected
- Proceeding directly to aggregation and analysis


### 1.2.2 Data Aggregation Strategy
create multiple dimensional views to support exploration and visualization.

- Keep `df_daily` as the canonical daily-grain table (Date, Time, State, Group, Unit, Sales).
- Build focused rollups for time and dimensional analysis: week/time, state/group, group/state, daily, weekly, monthly, quarterly.
- Each rollup will be reset-indexed and surfaced with shape and a sample for quick inspection.

In [ ]:
# Build canonical daily-grain and multiple aggregation views for analysis
required_cols = ['Date', 'Time', 'State', 'Group', 'Unit', 'Sales']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f'Missing required columns for aggregation: {missing}')

# df_daily: daily-grain (one row per Date, Time, State, Group)
df_daily = df.groupby(['Date', 'Time', 'State', 'Group'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
df_daily.reset_index(drop=True, inplace=True)
# Ensure Date is datetime for period-based rollups
df_daily['Date'] = pd.to_datetime(df_daily['Date'])
try:
    df_daily['Week'] = df_daily['Date'].dt.isocalendar().week
except Exception:
    df_daily['Week'] = df_daily['Date'].dt.week
df_daily['Month'] = df_daily['Date'].dt.month
df_daily['Quarter'] = df_daily['Date'].dt.quarter

# 1) Week-Time rollup
agg_time_week = df_daily.groupby(['Week', 'Time'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
print(f'agg_time_week: {agg_time_week.shape}')
print(agg_time_week.head())

# 2) State-Group rollup
agg_state_group = df_daily.groupby(['State', 'Group'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
print(f'agg_state_group: {agg_state_group.shape}')
print(agg_state_group.head())

# 3) Group-State rollup
agg_group_state = df_daily.groupby(['Group', 'State'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
print(f'agg_group_state: {agg_group_state.shape}')
print(agg_group_state.head())

# 4) Daily totals
agg_daily = df_daily.groupby(['Date'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
print(f'agg_daily: {agg_daily.shape}')
print(agg_daily.head())

# 5) Weekly totals
agg_weekly = df_daily.groupby(['Week'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
print(f'agg_weekly: {agg_weekly.shape}')
print(agg_weekly.head())

# 6) Monthly totals
agg_monthly = df_daily.groupby(['Month'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
print(f'agg_monthly: {agg_monthly.shape}')
print(agg_monthly.head())

# 7) Quarterly totals
agg_quarterly = df_daily.groupby(['Quarter'], as_index=False).agg({'Unit': 'sum', 'Sales': 'sum'})
print(f'agg_quarterly: {agg_quarterly.shape}')
print(agg_quarterly.head())

### Aggregation Strategy & GroupBy Insights

**Observation:** Each (Date, Time, State, Group) combination appears exactly once in the raw data.
This means we cannot collapse duplicates — instead, we strategically group by *subsets* of dimensions
to answer different business questions.

**Key insight:** GroupBy() is not for "merging duplicate rows" in this case, but for **dimensional analysis** —
selecting which dimensions to separate and which to collapse.

**Recommendation:** Store 7 aggregated views (one per analysis question) rather than recalculating
groupbys repeatedly in visualizations. This improves:
- Query clarity (each view has semantic meaning)
- Performance (aggregate once, use many times)
- Maintainability (single source of truth per analysis dimension).

### 1.3 Normalization

In [ ]:
# Placeholder for normalization logic — implement later
pass

## 2. Data Analysis
### 2.1 Descriptive Statistics

In [ ]:
# Placeholder for descriptive statistics (mean, median, mode, std)
# To be implemented later
pass

### 2.2 Highest and Lowest Sales

In [ ]:
# Placeholder for identifying highest/lowest by State and Group
pass

### 2.3 Time-Based Aggregations

In [ ]:
# Placeholder for weekly, monthly, quarterly rollups
pass

## 3. Data Visualization

### 3.1 State-wise Sales by Group

In [ ]:
# Placeholder for state-wise subplots
pass

### 3.2 Group-wise Sales by State

In [ ]:
# Placeholder for group-wise subplots
pass

### 3.3 Time-of-Day Analysis (Peak/Off-Peak)

In [ ]:
# Placeholder for time-of-day subplots
pass

## 4. Insights and Recommendations

[Placeholder for business insights]